# [Project title]

**Team members:** [Names]  
**Repository:** [URL of your team's fork]

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "The data directory is missing."

## Load data

In [ ]:
# Load data from DATA_DIR.
# load flu data
flu_df = pd.read_csv(DATA_DIR / "target-hospital-admissions.csv")

# inspect the data
display(flu_df.head())
    # note: preseve location as a string ( DON'T Perform math on it)
display(flu_df.info())
    # note: date, location, location_name -> strings
    # note: value, weekly_rate -> float
print("missing values in date: ", flu_df['date'].isnull().sum())
print("missing values in location: ", flu_df['location'].isnull().sum())
print("missing values in location_name: ", flu_df['location_name'].isnull().sum())
print("missing values in value: ", flu_df['value'].isnull().sum())
print("missing values in weekly_rate: ", flu_df['weekly_rate'].isnull().sum())
    # note: missing values in value ( 36) and missing values in weekly_rate ( 36)

print("Earliest date:\n ", flu_df["date"].min())
    # note: february 5, 2022
print("Latest date: \n", flu_df["date"].max())
    # note: october 31st, 2026

print("location names: \n",flu_df['location_name'].unique())
print("location names count: \n", flu_df['location_name'].nunique())
    # note: 53
    # discard: District of Columbia, US, and Puerto rico
    # keep : 50 states ( since focusing on the states in the U.S)

print("number of duplicate rows: \n", flu_df.duplicated().sum())
    # note: 0
print("number of duplicate date and location combination: \n",flu_df.duplicated(subset=["date", "location"]).sum())
    # note: 0 

# display the locations with missing hospitalization counts
print("locations with missing hospitalization counts:\n",flu_df[flu_df['value'].isna()]['location_name'].value_counts())
    # note: massachusetts, west virginia, minesota
# look at missing hospitalizations and the date, location_name, value, and weekly rate associated with it
print(flu_df[flu_df["value"].isna()][
    ["date", "location_name", "value", "weekly_rate"]

])
    # note: value and weekly_rate is missing for both
    # missing values occured around the same time as well ( ex: october 5, september 28 21 14 7, 
    # august 31 24 17 10 3, july 27, june 22 15 08 01,  may 25 18 all in 2024 )

# how many unique dates each locatation has
print(flu_df.groupby("location_name")["date"].nunique().describe())
    # each fo the 53 states has 248 dates

,date,location,location_name,value,weekly_rate
0,2026-10-31,37,North Carolina,202.0,1.828712
1,2026-10-31,42,Pennsylvania,557.0,4.258816
2,2026-10-31,04,Arizona,138.0,1.820008
3,2026-10-31,54,West Virginia,23.0,1.299450
4,2026-10-31,11,District of Columbia,10.0,1.423994


<class 'pandas.DataFrame'>
RangeIndex: 13144 entries, 0 to 13143
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           13144 non-null  str    
 1   location       13144 non-null  str    
 2   location_name  13144 non-null  str    
 3   value          13108 non-null  float64
 4   weekly_rate    13108 non-null  float64
dtypes: float64(2), str(3)
memory usage: 777.8 KB


None

missing values in date:  0
missing values in location:  0
missing values in location_name:  0
missing values in value:  36
missing values in weekly_rate:  36
Earliest date:
  2022-02-05
Latest date: 
 2026-10-31
location names: 
 <ArrowStringArray>
[      'North Carolina',         'Pennsylvania',              'Arizona',
        'West Virginia', 'District of Columbia',        'New Hampshire',
               'Nevada',         'Rhode Island',               'Alaska',
       'South Carolina',                 'Iowa',             'Maryland',
             'Colorado',             'Delaware',                'Maine',
                 'Ohio',              'Georgia',            'Minnesota',
            'Wisconsin',             'Oklahoma',              'Wyoming',
          'Mississippi',             'New York',          'Connecticut',
              'Indiana',        'Massachusetts',           'Washington',
              'Alabama',          'Puerto Rico',               'Hawaii',
               'Kansa

## Prepare data

In [56]:
# Clean data and create features.

# cleaning data
    # steps to do:
    # convert to date time
    # check if weekly date interval
    # handle missing values
    # filter out non-U.S states

# convert date to date time
flu_df["date"] = pd.to_datetime(flu_df["date"])

date_gaps = (
    flu_df.sort_values(["location", "date"])
    .groupby("location")["date"]
    .diff()
    .dt.days
)

print(date_gaps.value_counts().sort_index())
    # note: all 13,091 date intervals are 7 days apart

print("zero hospitilization count: ", (flu_df["value"] == 0).sum())
    # note: 1054
# check if any negative hospitilizations
print("negative hospitilization counts: ", (flu_df["value"]< 0).sum())
    # note: 0
display(flu_df["value"].describe())
    #13,108 non missing hospitilization values
    # mean = 224.60 average weekly count across locations
    # median = 20
        # could be right-skewed
    # max = largest weekly count 

# clean the data
flu_clean = flu_df.copy()
print(flu_clean.isna().sum())

    # leave missing rows since imputing missing to be unknown would
    # change the entire value and weekly_rate column to be a str rather than a numeric value

# convert to datetime
flu_clean["date"] = pd.to_datetime(flu_clean["date"])
print(flu_clean["date"].dtype)
print(flu_clean.dtypes)

# strip any white spaces
flu_clean["location_name"] = flu_clean["location_name"].str.strip()
flu_clean["location"] = flu_clean["location"].str.strip()

# only keep the 50 States in the US
sorted(flu_clean["location_name"].unique())

exclude_locations = ["District of Columbia", "Puerto Rico", "US"]
flu_clean = flu_clean.drop(
    flu_clean[flu_clean["location_name"].isin(exclude_locations)].index
)
# check
print("Rows:", len(flu_clean))
print("Location:", flu_clean["location_name"].nunique())

# save cleaned csv
flu_clean.to_csv(DATA_DIR / "flu_cleaned.csv", index=False)

date
7.0    13091
Name: count, dtype: int64
zero hospitilization count:  1054
negative hospitilization counts:  0


count    13108.000000
mean       224.596277
std       1585.864307
min          0.000000
25%          5.000000
50%         20.000000
75%         88.000000
max      55544.000000
Name: value, dtype: float64

date              0
location          0
location_name     0
value            36
weekly_rate      36
dtype: int64
datetime64[us]
date             datetime64[us]
location                    str
location_name               str
value                   float64
weekly_rate             float64
dtype: object
Rows: 12400
Location: 50


## Explore data

In [ ]:
# Summarize and visualize the prepared data.

## Build models

In [ ]:
# Train baseline and candidate models.

## Evaluate models

In [ ]:
# Compare models using the chosen evaluation measures.

## Final results

In [ ]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]